<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Position and masking

**[Position and masking](https://learning.nextia-ai.com/courses/deep-learning/m06/position-and-masking/)** · Deep Learning and Transformers Explained · Module 6, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** show that attention ignores word order, add position vectors, and use the padding mask and the causal mask on real tensors.

| | |
|---|---|
| **Time** | About 35 minutes. The training cell takes one to three minutes. |
| **Needs** | A browser and a free Colab or Kaggle account. The setup cells download the tickets (about 2 MB) and the course's `ticketnet.py`. In Kaggle, turn on Internet in the notebook settings. |
| **You should know** | The transformer block, from [Transformer blocks](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/). |
| **Tested** | 2026-10-08, Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m06/position-and-masking/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M06-L02-position-and-masking/position-and-masking-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: train a tiny transformer

Sections 2 to 4 need a trained model. The next cell trains `TinyTransformer` for 8 epochs, with seed 0. The lesson used 20 epochs, so your scores differ a little from the lesson page. Run the cell. It takes one to three minutes.

In [ ]:
from torch import nn

vocab = Vocab([row["text"] for row in train_rows])
ids_train, ids_valid = token_ids(train_rows, vocab), token_ids(valid_rows, vocab)
y_train, y_valid = labels(train_rows), labels(valid_rows)
set_seed(0)
model = TinyTransformer(len(vocab))
history = train(model, ids_train, y_train, ids_valid, y_valid, epochs=8, lr=1e-3, log=None)
model.eval()
print("valid accuracy after 8 epochs:", history[-1]["valid_acc"])

> **Check.** You should see a validation accuracy of about 0.92.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The shuffle test

Two made-up sentences have the same 7 tokens in a different order. One attention layer reads each one, and the average of its outputs is a summary.

> **Predict.** Are the two summaries the same? Run the cell.

In [ ]:
words = ["damaged", "on", "arrival", ",", "not", "after", "use"]
a = [0, 1, 2, 3, 4, 5, 6]          # damaged on arrival , not after use
b = [0, 5, 6, 3, 4, 1, 2]          # damaged after use , not on arrival
torch.manual_seed(0)
table = nn.Embedding(len(words), 4)
attention = nn.MultiheadAttention(4, 1, batch_first=True)

def read(order):
    x = table(torch.tensor([order]))
    out, _ = attention(x, x, x)
    return out.mean(dim=1).detach()

print(read(a))
print(read(b))
print(torch.allclose(read(a), read(b), atol=1e-6))

> **Check.** You should see the same 4 numbers twice and `True`.

Now add a position vector to each word vector. Run the cell.

In [ ]:
torch.manual_seed(1)
position = nn.Embedding(7, 4)
steps = torch.arange(7)

def read_in_order(order):
    x = table(torch.tensor([order])) + position(steps)   # word vector + position vector
    out, _ = attention(x, x, x)
    return out.mean(dim=1).detach()

print(torch.allclose(read_in_order(a), read_in_order(b), atol=1e-6))

> **Check.** You should see `False`: with positions, the order changes the result.

## 2. The trained model without positions

The cell shuffles the tokens of ticket T-64697, then compares the scores with the position table and with the table set to zero.

> **Predict.** With the position table at zero, will the shuffled ticket get the same 5 scores? Run the cell.

In [ ]:
import random

ticket = valid_rows[0]["text"]
tokens = tokenize(ticket)
random.seed(0)
shuffled = " ".join(random.sample(tokens, len(tokens)))

def scores(text):
    with torch.no_grad():
        return [round(v, 3) for v in model(torch.tensor([vocab.encode(text)]))[0].tolist()]

print("with positions:   ", scores(ticket), scores(shuffled))
saved = model.position.weight.data.clone()
model.position.weight.data.zero_()
print("without positions:", scores(ticket), scores(shuffled))
model.position.weight.data = saved

> **Check.** With positions, the two lists differ. Without positions, they are equal.

## 3. The padding mask

The cell puts a short delivery ticket in a batch with the longest validation ticket, and runs the model's steps with and without the padding mask. The teams are in the order delivery, returns, payment, warranty, account.

> **Predict.** Which two lines will be equal? Run the cell.

In [ ]:
short = next(row for row in valid_rows if row["style"] == "short")
long = max(valid_rows, key=lambda row: len(vocab.encode(row["text"])))
pair = token_ids([short, long], vocab)
print(short["ticket_id"], short["team"], "| padding tokens:", int((pair[0] == PAD).sum()))

def team_scores(ids, use_mask):
    with torch.no_grad():
        h = model.embed(ids) + model.position(torch.arange(ids.shape[1]))
        mask = ids == PAD if use_mask else None
        h = model.blocks(h, src_key_padding_mask=mask)
        keep = (ids != PAD).unsqueeze(-1).float()
        mean = (model.norm(h) * keep).sum(1) / keep.sum(1)
        return [round(v, 3) for v in model.out(mean)[0].tolist()]

print("alone, no padding:   ", team_scores(token_ids([short], vocab), use_mask=True))
print("padded, with mask:   ", team_scores(pair, use_mask=True))
print("padded, without mask:", team_scores(pair, use_mask=False))

> **Check.** You should see 107 padding tokens. The first two lines are equal; the third differs.

## 4. The causal mask

`nn.Transformer.generate_square_subsequent_mask(n)` makes the causal mask: 0 where a token may read, minus infinity where it may not.

> **Your turn.** Make `causal` for 6 tokens, and put the number of hidden cells in `hidden_cells`. `torch.isinf()` is `True` where a value is infinite. Predict the number first. Then run the check cell.

In [ ]:
causal = ...
hidden_cells = ...

In [ ]:
expect_hash('hidden_cells', hidden_cells, 'e629fa6598d73276')

The mask on one attention head: every weight above the diagonal is 0. Run the cell.

In [ ]:
torch.manual_seed(0)
x = torch.randn(1, 6, 8)
head = nn.MultiheadAttention(8, 1, batch_first=True)
_, weights = head(x, x, x, attn_mask=nn.Transformer.generate_square_subsequent_mask(6))
print(weights[0].detach().numpy().round(2))

> **Check.** The first row should be `1.` and five zeros, and the upper triangle should be all 0.

## 5. Failure case: a mask turned the wrong way

In PyTorch, `True` in a mask means **hide**. Tomás writes the tokens to keep, `ids != PAD`.

> **Predict.** What happens to the accuracy, and to a ticket with no padding at all? Run the cell.

In [ ]:
def scores_with(mask):
    with torch.no_grad():
        h = model.embed(ids_valid) + model.position(torch.arange(ids_valid.shape[1]))
        h = model.blocks(h, src_key_padding_mask=mask)
        keep = (ids_valid != PAD).unsqueeze(-1).float()
        return model.out((model.norm(h) * keep).sum(1) / keep.sum(1))

wrong = scores_with(ids_valid != PAD)
right = scores_with(ids_valid == PAD)
print("wrong mask: accuracy", round((wrong.argmax(1) == y_valid).float().mean().item(), 4),
      "| tickets with nan scores:", int(torch.isnan(wrong).any(1).sum()))
print("right mask: accuracy", round((right.argmax(1) == y_valid).float().mean().item(), 4))

> **Check.** The wrong mask gives a low accuracy and 23 tickets with `nan` scores: the tickets of 120 tokens, where every key is hidden. The right mask gives about 0.92.

## Next

You showed that attention ignores order without positions, and you used both masks. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m06/position-and-masking/) has the causal-mask diagram, the mask explorer and the knowledge checks. Next: how a model learns from text with no labels.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Training objectives](https://learning.nextia-ai.com/courses/deep-learning/m06/training-objectives/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m06/position-and-masking/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.